## Problem Statement

### Business Context

As organizations grow, business analysts increasingly face the challenge of navigating large volumes of reports, research papers, and strategic documents. Extracting the right insights from lengthy materials can be time-consuming and overwhelming, especially when these insights directly influence key business decisions.

Consider joining a venture capital firm like Andreessen Horowitz and being assigned a dense report such as Harvard Business Review’s **“How Apple is Organized for Innovation.”** Manually reviewing such documents requires significant effort, slowing down the analysis process and increasing the chances of missing important details.

To overcome this information overload, businesses can leverage **Semantic Search** and **Retrieval-Augmented Generation (RAG)** models. These systems allow analysts to ask natural-language questions like, *“How does Apple structure its teams for innovation?”* and instantly retrieve relevant, accurate insights from the source document.

By integrating such AI-driven retrieval systems, organizations can streamline research workflows, reduce manual effort, and enable analysts to focus on high-value strategic thinking, ultimately improving decision-making speed and quality.


**Common Questions to Answer**

1. Who are the authors of this article and who published this article?

2. List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines.

3. Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?



### Objective

As an AI specialist, your task is to develop a RAG-based application that enables business analysts to efficiently extract insights from extensive business reports such as “How Apple is Organized for Innovation.” The objective is to understand the challenges of navigating long, information-dense documents, apply retrieval-augmented generation techniques to surface only the most relevant content, analyze how this improves the speed and accuracy of report interpretation, evaluate its potential to enhance strategic decision-making and productivity for analysts, and create a functional prototype that demonstrates the system’s effectiveness in answering queries, summarizing key insights, and supporting natural-language interactions without requiring users to read the entire report.

### Data Description

**“How Apple is Organized for Innovation”** is a detailed Harvard Business Review article that examines Apple’s unique approach to structuring teams, driving innovation, and maintaining a culture of excellence. The article is provided as a PDF consisting of **11 pages**, offering in-depth insights into Apple’s organizational design, leadership principles, and decision-making processes.


## **Please read the instructions carefully before starting the project.**

This is a commented Python Notebook file in which all the instructions and tasks to be performed are mentioned.
* Blanks '_____' are provided in the notebook that
needs to be filled with an appropriate code to get the correct result. With every '_____' blank, there is a comment that briefly describes what needs to be filled in the blank space.
* Identify the task to be performed correctly, and only then proceed to write the required code.
* Please run the codes in a sequential manner from the beginning to avoid any unnecessary errors.
* Add the results/observations (wherever mentioned) derived from the analysis in the presentation and submit the same. Any mathematical or computational details which are a graded part of the project can be included in the Appendix section of the presentation.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Install required libraries
!pip install -q langchain_community==0.3.27 \
              langchain==0.3.27 \
              chromadb==1.0.15 \
              pymupdf==1.26.3 \
              tiktoken==0.9.0 \
              datasets==4.0.0 \
              evaluate==0.4.5 \
              langchain_openai==0.3.30

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 459.1/459.1 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 k

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [1]:
# Import core libraries
import os                                                                       # Interact with the operating system (e.g., set environment variables)
import json                                                                     # Read/write JSON data
import requests  # type: ignore                                                 # Make HTTP requests (e.g., API calls); ignore type checker

# Import libraries for working with PDFs and OpenAI
from langchain.document_loaders import PyMuPDFLoader                            # Load and extract text from PDF files
# from langchain_community.document_loaders import PyPDFLoader                    # Load and extract text from PDF files
from openai import OpenAI                                                       # Access OpenAI's models and services

# Import libraries for processing dataframes and text
import tiktoken                                                                 # Tokenizer used for counting and splitting text for models
import pandas as pd                                                             # Load, manipulate, and analyze tabular data

# Import LangChain components for data loading, chunking, embedding, and vector DBs
from langchain.text_splitter import RecursiveCharacterTextSplitter              # Break text into overlapping chunks for processing
from langchain.embeddings.openai import OpenAIEmbeddings                        # Create vector embeddings using OpenAI's models  # type: ignore
from langchain.vectorstores import Chroma                                       # Store and search vector embeddings using Chroma DB  # type: ignore

from datasets import Dataset                                                    # Used to structure the input (questions, answers, contexts etc.) in tabular format
from langchain_openai import ChatOpenAI                                         # This is needed since LLM is used in metric computation

## Question Answering using LLM

### OpenAI API Calling



> **Note:** <br> Make sure to create a `config.json` file in your project directory containing your OpenAI credentials in the following format:
<br><br>```{"API_KEY": "your_openai_api_key_here","OPENAI_API_BASE": "your_api_base"}```<br><br>
Replace the placeholder with your actual API key. This file allows your script to securely load API configuration details without hardcoding them directly into the code. </br>


In [2]:
# Uncomment the below snippet of code if the drive needs to be mounted
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Load the JSON file and extract values
file_name = "/content/drive/MyDrive/UT Agents/Module 1/Project/config"                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    OPENAI_API_KEY = config.get("API_KEY")                                             # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")                             # Extract the OpenAI base URL from the config

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY                                          # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE                                 # Set API base URL as environment variable

# Initialize OpenAI client
client = OpenAI()

In [4]:
print(f"OPENAI_API_KEY: {os.environ.get('OPENAI_API_KEY')}")
print(f"OPENAI_BASE_URL: {os.environ.get('OPENAI_BASE_URL')}")

OPENAI_API_KEY: gl-U2FsdGVkX19lCJEzDl6Xi2bnn/JZxyZ0h7+9/VRDDx3ERZ8sbPCcz0EnNyBv3qbb
OPENAI_BASE_URL: https://aibe.mygreatlearning.com/openai/v1


### Response

In [5]:
# Define a function to get a response
def response(user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):   # Complete the code to set default paramenters
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o-mini",  # Complete the code by specifying the model to be used.
        messages=[
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content                                # Return the text content from the model's reply

### Question 1: Who are the authors of this article and who published this article?

In [6]:
question_1 = "Who are the authors of this article and who published this article ?"
base_prompt_response_1=response(question_1)

### Question 2: List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines.

In [7]:
question_2 = " List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines" #Complete the code to define the question #2
base_prompt_response_2=response(question_2) #Complete the code to pass the user input

### Question 3: Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?

In [8]:
question_3 = "Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?" #Complete the code to define the question #3
base_prompt_response_3=response(question_3) #Complete the code to pass the user input

In [9]:
print(f"Base Prompt Response 1:\n{base_prompt_response_1}")
print(f"\nBase Prompt Response 2:\n{base_prompt_response_2}")
print(f"\nBase Prompt Response 3:\n{base_prompt_response_3}")

Base Prompt Response 1:
I'm sorry, but I can't access external content, including specific articles or their authors. If you provide me with the title or more details about the article, I may be able to help you with general information or context related to the topic.

Base Prompt Response 2:
- **Vision**: Effective leaders possess a clear and compelling vision for the future, inspiring their team to work towards common goals. This foresight helps guide decision-making and aligns efforts across the organization.

- **Empathy**: Strong leaders demonstrate empathy by understanding and valuing the perspectives and emotions of their team members. This fosters trust and collaboration, creating a supportive work environment where individuals feel valued.

- **Integrity**: Leaders with integrity uphold ethical standards and are transparent in their actions and decisions. This builds credibility and trust among team members, encouraging a culture of honesty and accountability within the organ

**Observations:**
-For Q1, the model honestly declined to answer, stating it cannot access external articles — it had no knowledge of this specific document and returned no useful output
For Q2, the model generated entirely generic leadership characteristics (Vision, Empathy, Integrity) drawn from its general training knowledge, none of which match the actual three characteristics described in the article — a clear case of hallucination
For Q3, the model produced confident-sounding but fabricated Apple examples (Steve Jobs, iPhone, Apple Watch Watch) from its training data rather than the article — the answers sounded plausible but were not grounded in the source document
Without any document context, the base LLM cannot reliably answer document-specific questions and either refuses or hallucinates
-

## Question Answering using LLM with Prompt Engineering

In the next step, we will use prompt engineering to check the effect of a more detailed and well-engineered prompt on the output of the model.

In [10]:
system_prompt = """
You are an AI assistant specializing in business analysis and strategic insights. Your role is to extract clear, precise, and accurate insights from business reports, research papers, and strategic documents, such as the 'How Apple is Organized for Innovation' article.

When answering, prioritize factual correctness based on the provided document, ensure clarity for business analysts, and focus on details relevant to organizational design, innovation, and leadership.

If the requested information is not explicitly available in the provided document, state that the information cannot be found in the given context rather than speculating or generating external information.
"""

### Defining the function to Generate a Response From the LLM

In [11]:
# Define a function to get a response from the OpenAI chat model
def response(system_prompt, user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):  # Complete the code to set default paramenters
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o-mini",                                                        # Complete the code by specifying the model to be used.
        messages=[
            {"role": "system", "content": system_prompt},                       # System prompt sets the assistant's behavior
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output (0 = deterministic)
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content                                # Return the text content from the model's reply

### Question 1: Who are the authors of this article and who published this article ?

In [12]:
response_with_prompt_eng_1=response(system_prompt,question_1)
response_with_prompt_eng_1

'The article "How Apple is Organized for Innovation" is authored by David W. McMillan and was published by the Harvard Business Review.'

### Question 2: List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines.

In [13]:
response_with_prompt_eng_2=response(system_prompt,question_2) #Complete the code to pass the user prompt and system prompt
response_with_prompt_eng_2

"The three leadership characteristics highlighted in the 'How Apple is Organized for Innovation' article are:\n\n- **Visionary Thinking**  \n  Leaders at Apple possess a clear and compelling vision that guides the organization’s direction, inspiring innovation and aligning teams towards common goals.\n\n- **Collaborative Mindset**  \n  Apple leaders foster a culture of collaboration, encouraging cross-functional teamwork that enhances creativity and accelerates the innovation process by leveraging diverse perspectives.\n\n- **Decisive Action**  \n  Effective leaders at Apple make timely and informed decisions, ensuring that strategic initiatives are executed promptly to maintain momentum in the fast-paced technology landscape."

### Question 3: Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?

In [14]:
response_with_prompt_eng_3=response(system_prompt,question_3) #Complete the code to pass the user prompt and system prompt
response_with_prompt_eng_3

'I’m sorry, but I cannot provide specific examples from the article "How Apple is Organized for Innovation" as I do not have access to its content. Therefore, I cannot detail instances where Apple\'s approach to leadership has resulted in successful innovations. If you have particular excerpts or sections from the article you\'d like to discuss, please share them, and I can help analyze those.'

In [15]:
print(f"\Response 1 with Prompt Engineering:\n{response_with_prompt_eng_1}")
print(f"\Response 2 with Prompt Engineering:\n{response_with_prompt_eng_2}")
print(f"\Response 3 with Prompt Engineering:\n{response_with_prompt_eng_3}")

\Response 1 with Prompt Engineering:
The article "How Apple is Organized for Innovation" is authored by David W. McMillan and was published by the Harvard Business Review.
\Response 2 with Prompt Engineering:
The three leadership characteristics highlighted in the 'How Apple is Organized for Innovation' article are:

- **Visionary Thinking**  
  Leaders at Apple possess a clear and compelling vision that guides the organization’s direction, inspiring innovation and aligning teams towards common goals.

- **Collaborative Mindset**  
  Apple leaders foster a culture of collaboration, encouraging cross-functional teamwork that enhances creativity and accelerates the innovation process by leveraging diverse perspectives.

- **Decisive Action**  
  Effective leaders at Apple make timely and informed decisions, ensuring that strategic initiatives are executed promptly to maintain momentum in the fast-paced technology landscape.
\Response 3 with Prompt Engineering:
I’m sorry, but I cannot pro

<>:1: SyntaxWarning: invalid escape sequence '\R'
<>:2: SyntaxWarning: invalid escape sequence '\R'
<>:3: SyntaxWarning: invalid escape sequence '\R'
<>:1: SyntaxWarning: invalid escape sequence '\R'
<>:2: SyntaxWarning: invalid escape sequence '\R'
<>:3: SyntaxWarning: invalid escape sequence '\R'
/tmp/ipykernel_1986/3164977487.py:1: SyntaxWarning: invalid escape sequence '\R'
  print(f"\Response 1 with Prompt Engineering:\n{response_with_prompt_eng_1}")
/tmp/ipykernel_1986/3164977487.py:2: SyntaxWarning: invalid escape sequence '\R'
  print(f"\Response 2 with Prompt Engineering:\n{response_with_prompt_eng_2}")
/tmp/ipykernel_1986/3164977487.py:3: SyntaxWarning: invalid escape sequence '\R'
  print(f"\Response 3 with Prompt Engineering:\n{response_with_prompt_eng_3}")


**Observations**:
For Q1, the model hallucinated an entirely incorrect author name (David W. McMillan) despite the system prompt instructing it not to speculate — a system prompt alone cannot prevent factual errors when the document is not provided
For Q2, the model returned plausible but incorrect leadership characteristics (Visionary Thinking, Collaborative Mindset, Decisive Action) — these sound reasonable but are not the actual three from the article (Deep Expertise, Immersion in Details, Willingness to Collaboratively Debate)
For Q3, the system prompt worked as intended — the model correctly declined to answer, stating it had no access to the article content rather than fabricating examples as it did without the system prompt
Prompt engineering improved behavior for Q3 (reduced hallucination) but was insufficient for Q1 and Q2 — factual accuracy still requires the actual document content-
-


## Data Preparation for RAG

### Loading the Data

In [ ]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [40]:
pdf_path = "/content/drive/MyDrive/UT Agents/Module 1/Project/HBR_How_Apple_Is_Organized_For_Innovation.pdf" #Complete the code to define the file name

In [41]:
pdf_loader = PyMuPDFLoader(pdf_path)

In [42]:
pdf = pdf_loader.load()

### Data Overview

#### Checking the first 5 pages

In [43]:
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(pdf[i].page_content,end="\n")

Page Number : 1
REPRINT R2006F
PUBLISHED IN HBR
NOVEMBER–DECEMBER 2020
ARTICLE
ORGANIZATIONAL CULTURE
How Apple Is 
Organized  
for Innovation
It’s about experts leading experts. 
by Joel M. Podolny and Morten T. Hansen
This article is made available to you with compliments of Apple Inc for your personal use. Further posting, copying or distribution is not permitted.
Page Number : 2
2
Harvard Business Review
November–December 2020
This article is made available to you with compliments of Apple Inc for your personal use. Further posting, copying or distribution is not permitted.
Page Number : 3
PHOTOGRAPHER MIKAEL JANSSON
How Apple Is 
Organized 
for Innovation
It’s about experts 
leading experts.
ORGANIZATIONAL 
CULTURE
Joel M. 
Podolny
Dean, Apple 
University
Morten T. 
Hansen
Faculty, Apple 
University
AUTHORS
FOR ARTICLE REPRINTS CALL 800-988-0886 OR 617-783-7500, OR VISIT HBR.ORG
Harvard Business Review
November–December 2020  3
This article is made available to you with compliment

### Data Chunking

#### Chunk the PDF into Manageable Text Sections Using a Token-Based Splitter

In [44]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=1024, # Increased chunk size
    chunk_overlap=200
)

#### Split the Loaded PDF into Chunks for Further Processing

In [67]:
#document_chunks = pdf_loader.load_and_split(text_splitter)

# Load all pages
pages = pdf_loader.load()

# Merge into a single document so overlap bridges page boundaries
from langchain.schema import Document
combined_doc = [Document(
    page_content="\n".join([p.page_content for p in pages]),
    metadata={'source': pdf_path}
)]

# Now split — overlap=200 will properly span across pages
document_chunks = text_splitter.split_documents(combined_doc)

#### Check the Number of Chunks Created

In [68]:
len(document_chunks)

11

### Generate Vector Embeddings for Text Chunks Using OpenAI

In [69]:
# Initialize the OpenAI Embeddings model with API credentials
embedding_model = OpenAIEmbeddings(
    openai_api_key=OPENAI_API_KEY,                                                     # Your OpenAI API key for authentication
    openai_api_base=OPENAI_API_BASE                                             # The OpenAI API base URL endpoint
)

# Generate embeddings (vector representations) for the first two document chunks
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)      # Embedding for chunk 0
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)      # Embedding for chunk 1

# Check and print the dimension (length) of the embedding vector
print("Dimension of the embedding vector ", len(embedding_1))                   # Typically 1536 or 2048 depending on model

Dimension of the embedding vector  1536


In [70]:
# Verify if both embeddings have the same dimension (should be True)
len(embedding_1) == len(embedding_2)

# Return/display the two embedding vectors for further inspection or use
embedding_1, embedding_2

([0.008166258126992627,
  -0.0031960699330132072,
  -0.008212770238465228,
  -9.276535165230237e-05,
  -0.0018737873539875652,
  0.02966165507601661,
  -0.00966794591164701,
  0.022126639000746504,
  -0.002486755110781937,
  -0.020997050901840623,
  0.0015116546823149871,
  0.00776757989226209,
  -0.011734427745754335,
  -0.0013970347189337876,
  -0.02543566718485676,
  0.02393398033152494,
  0.019455494129559978,
  -0.011674625661298794,
  -0.01227928776705543,
  -0.004721013773403892,
  -0.03973492560586948,
  0.007255942475111938,
  -0.009309135267558886,
  0.0014510224402092932,
  -0.0006034159961665044,
  0.015521869811541238,
  0.01661159078546599,
  -0.02692406592785077,
  0.02958192082605435,
  0.007614753119200063,
  0.008657961050329649,
  0.01606672983284233,
  -0.019973776998862883,
  0.0015980349665066036,
  -0.012970330040588361,
  -0.011661336619638417,
  -0.009136374932006294,
  -0.023575169687436816,
  0.0079403397621534,
  -0.034286323530213415,
  0.03447237197610382,

### Vector Database

#### Setup Vector Database Directory

In [71]:
out_dir = '/content/chroma_db'    # complete the code to define the name of the vector database

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

#### Create Vector Database from Documents

In [72]:
# Building the vector store and saving it to disk for future use
vectorstore = Chroma.from_documents(
    document_chunks,                                                            # Documents to index
    embedding_model,                                                            # Embedding model for converting text to vectors
    persist_directory=out_dir                                                   # Save vector DB files here
)

#### Load Vector Database

In [73]:
vectorstore = Chroma(
    persist_directory=out_dir,
    embedding_function=embedding_model
)

#### Explore Vector Database and Perform Searches

In [74]:
vectorstore.embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x7f1894a481a0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7f1894a4a3c0>, model='text-embedding-ada-002', deployment='text-embedding-ada-002', openai_api_version='', openai_api_base='https://aibe.mygreatlearning.com/openai/v1', openai_api_type='', openai_proxy='', embedding_ctx_length=8191, openai_api_key='gl-U2FsdGVkX19lCJEzDl6Xi2bnn/JZxyZ0h7+9/VRDDx3ERZ8sbPCcz0EnNyBv3qbb', openai_organization=None, allowed_special=set(), disallowed_special='all', chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None)

write an instruction on what to do in next cell

In [75]:
vectorstore.similarity_search("Apple's organizational design for innovation",k=3) #Complete the code to pass a query and an appropriate k value

[Document(metadata={'encryption': 'Standard V2 R3 128-bit RC4', 'page': 2, 'subject': '', 'keywords': '', 'modDate': 'D:20201201183749Z', 'creator': 'Adobe InDesign 14.0 (Macintosh)', 'title': '', 'creationdate': '2020-10-05T14:18:42-04:00', 'trapped': '', 'creationDate': "D:20201005141842-04'00'", 'author': '', 'file_path': '/content/drive/MyDrive/UT Agents/Module 1/Project/HBR_How_Apple_Is_Organized_For_Innovation.pdf', 'source': '/content/drive/MyDrive/UT Agents/Module 1/Project/HBR_How_Apple_Is_Organized_For_Innovation.pdf', 'moddate': '2020-12-01T18:37:49+00:00', 'format': 'PDF 1.6', 'producer': 'Adobe PDF Library 15.0 (via http://bfo.com/products/pdf?version=2.23.5-r33279)', 'total_pages': 11}, page_content='PHOTOGRAPHER\u2002MIKAEL JANSSON\nHow Apple Is \nOrganized \nfor Innovation\nIt’s about experts \nleading experts.\nORGANIZATIONAL \nCULTURE\nJoel M. \nPodolny\nDean, Apple \nUniversity\nMorten T. \nHansen\nFaculty, Apple \nUniversity\nAUTHORS\nFOR ARTICLE REPRINTS CALL 800-9

### Retrieval and Response Generation using Vector Search

#### Convert Vector Database into a Retriever and Retrieve Relevant Documents

In [76]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 8} #Complete the code to pass an appropriate k value
)

### System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:

    1. The system message describing the assistant's role.
    2. A user message template including context and the question.

In [77]:
qna_system_message = """
You are an AI assistant specializing in business analysis and strategic insights. Your primary role is to extract clear, precise, and accurate insights directly from provided business reports and articles, such as "How Apple is Organized for Innovation."

When answering, adhere strictly to the following guidelines:
1.  **Use Only Provided Context:** Base your answers *solely* on the information explicitly contained within the provided document excerpts.
2.  **Factual Correctness & Clarity:** Ensure factual accuracy and present insights clearly, focusing on details relevant to organizational design, innovation, and leadership.
3.  **Handle Missing Information:** If the requested information is not explicitly available in the provided context, state clearly that "The information cannot be found in the provided context." Do not speculate or introduce external knowledge.

Avoid:
- Summarizing or rephrasing information unless directly required by the question.
- Generating any information not present in the provided context.
"""

In [78]:
qna_user_message_template = """
### Context:
The following excerpts from the document are provided to help answer the question:
{context}

### Question:
{question}
"""

### Response Function

In [79]:
def generate_rag_response(user_input,k=3,max_tokens=500,temperature=0,top_p=0.95):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    # --- Debugging Print Statements ---
    print(f"\n--- Context for '{user_input}' ---")
    print(context_for_query)
    print(f"\n--- Full User Message for '{user_input}' ---")
    print(user_message)
    print("-------------------------------------")
    # -----------------------------------

    # Generate the response
    try:
        response = client.chat.completions.create(
        model="gpt-4o-mini",   # Complete the code by specifying the model to be used.
        messages=[
            {"role": "system", "content": qna_system_message},
            {"role": "user", "content": user_message}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p
        )
        # Extract and print the generated text from the response
        response = response.choices[0].message.content.strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Question 1: Who are the authors of this article and who published this article ?

In [80]:
response_with_rag_1 = generate_rag_response(question_1, k=8)
response_with_rag_1


--- Context for 'Who are the authors of this article and who published this article ?' ---
2
Harvard Business Review
November–December 2020
This article is made available to you with compliments of Apple Inc for your personal use. Further posting, copying or distribution is not permitted.. 2
Harvard Business Review
November–December 2020
This article is made available to you with compliments of Apple Inc for your personal use. Further posting, copying or distribution is not permitted.. PHOTOGRAPHER MIKAEL JANSSON
How Apple Is 
Organized 
for Innovation
It’s about experts 
leading experts.
ORGANIZATIONAL 
CULTURE
Joel M. 
Podolny
Dean, Apple 
University
Morten T. 
Hansen
Faculty, Apple 
University
AUTHORS
FOR ARTICLE REPRINTS CALL 800-988-0886 OR 617-783-7500, OR VISIT HBR.ORG
Harvard Business Review
November–December 2020  3
This article is made available to you with compliments of Apple Inc for your personal use. Further posting, copying or distribution is not permitted.. PHOTOGRAPHE

'The authors of the article are Joel M. Podolny and Morten T. Hansen. The article is published by Harvard Business Review.'

### Question 2: List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines.

In [81]:
response_with_rag_2 = generate_rag_response(question_2) #Complete the code to pass the user input
response_with_rag_2


--- Context for ' List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines' ---
technologies and designs are likely to succeed in smart-
phones, computers, and so on. Relying on technical experts 
rather than general managers increases the odds that those 
bets will pay off.
Second, Apple’s commitment to offer the best possible 
products would be undercut if short-term profit and cost 
ABOUT THE ART
Apple Park, Apple’s corporate headquarters in  
Cupertino, California, opened in 2017.
Mikael Jansson/Trunk Archive
FOR ARTICLE REPRINTS CALL 800-988-0886 OR 617-783-7500, OR VISIT HBR.ORG
Harvard Business Review
November–December 2020  5
This article is made available to you with compliments of Apple Inc for your personal use. Further posting, copying or distribution is not permitted.
targets were the overriding criteria for judging investments 
and leaders. Significantly, the bonuses of senior R&D exec-
utives are based

'- **Deep Expertise:** Managers are expected to have specialized knowledge in their respective functions, allowing them to engage meaningfully in the work being done.\n\n- **Immersion in Details:** Leaders are required to be deeply involved in the specifics of their functions, ensuring informed decision-making based on comprehensive understanding.\n\n- **Collaborative Debate:** Managers must be willing to engage in discussions across functions, fostering coordinated decision-making by leveraging the expertise of all relevant parties.'

### Question 3: Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?

In [82]:
response_with_rag_3 = generate_rag_response(question_3, k=8) #Complete the code to pass the user input
response_with_rag_3


--- Context for 'Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?' ---
specialization. Apple’s leaders believe that world-class talent 
wants to work for and with other world-class talent in a 
specialty. It’s like joining a sports team where you get to learn 
from and play with the best.
Early on, Steve Jobs came to embrace the idea that 
managers at Apple should be experts in their area of man-
agement. In a 1984 interview he said, “We went through that 
stage in Apple where we went out and thought, Oh, we’re 
gonna be a big company, let’s hire professional management. 
We went out and hired a bunch of professional management. 
It didn’t work at all....They knew how to manage, but they 
didn’t know how to do anything. If you’re a great person, why 
do you want to work for somebody you can’t learn anything 
from? And you know what’s interesting? You know who the 
best managers are? They are the great individual

'One specific example from the article highlighting Apple\'s approach to leadership leading to successful innovations is the introduction of the dual-lens camera with portrait mode in the iPhone 7 Plus in 2016. This innovation was a significant risk, as it involved a substantial investment in a more costly camera. Paul Hubel, a senior leader involved in this project, was described as being "out over his skis," indicating that the team was taking a big gamble on whether users would be willing to pay a premium for the enhanced camera features. The success of this innovation not only defined the iPhone 7 Plus but also enhanced the reputations of Hubel and his team, demonstrating how deep expertise and a willingness to take calculated risks can lead to successful product innovations.\n\nAnother example is the leadership of Roger Rosner, who heads Apple’s software application business. His deep expertise in software engineering and previous experience allowed him to effectively lead a team 

### **Observations**

For Q1, the model correctly identified both authors (Joel M. Podolny and Morten T. Hansen) and the publisher (Harvard Business Review) by retrieving the exact information directly from the document
For Q2, the model accurately described all three leadership characteristics exactly as stated in the article — Deep Expertise, Immersion in Details, and Collaborative Debate — with correct explanations grounded in the source text
For Q3, after fixing the chunking strategy (merging pages before splitting so overlap bridged page boundaries), the model successfully retrieved and explained specific innovation examples from the article, including the portrait mode development story
RAG consistently outperformed both the base LLM and prompt-engineered LLM across all three questions, confirming that grounding generation in retrieved document context is essential for accurate, document-specific question answering-
-
-

## Actionable Insights and Business Recommendations


*   
*  
*



<font size=6 color='#4682B4'>Power Ahead</font>
___